# Seguindo Estratégias do EDA_2024

Justificação das Transformações de Dados (EDA)
1. Criação da flag Tem_Ingles antes da imputação de zeros

Porquê: Em Machine Learning e análise estatística, existe uma diferença abismal entre "o aluno teve nota zero na prova" e "o aluno não teve a disciplina". Se imputarmos diretamente o valor 0 nos nulos (NaN), o modelo interpretará que todos esses alunos falharam redondamente no Inglês. Ao criar a variável categórica Tem_Ingles (1 para sim, 0 para não), preservamos a realidade estrutural do currículo da ONG, ensinando ao modelo que a ausência de nota se deve à não obrigatoriedade da disciplina naquelas fases específicas.

2. Recriação dos Rankings (CG_2023, CF_2023 e CT_2023)

Porquê: As colunas originais de Classificação Geral (CG), Classificação por Fase (CF) e Classificação por Turma (CT) estavam vazias. No entanto, o ranking nada mais é do que a ordenação da nota final. Ao reconstruir estas métricas com base no INDE 2023, recuperamos atributos valiosos de posicionamento relativo. Isto permite avaliar o desempenho de um aluno não apenas pela sua nota absoluta, mas pela sua posição competitiva dentro do seu grupo de pares. (Nota: as colunas foram ajustadas para o sufixo 2023 para manter a coerência do ano analisado).

3. Eliminação de colunas 100% nulas e renomeação de variáveis reais

Porquê: A exclusão das colunas de recomendações (Rec Av1, Rec Psicologia, etc.), destaques e duplicados vazios (INDE 23, Pedra 23) é um passo fundamental de redução de dimensionalidade. Variáveis com 100% de valores nulos não contêm variância e apenas introduzem "ruído" computacional e visual. Em simultâneo, renomear as colunas válidas (ex: INDE 2023 para INDE_23) elimina artefactos de exportação do sistema e garante a integridade nas futuras cruzas de dados.

4. Definição de Threshold (corte) para o Indicador IPV

Porquê: O "Ponto de Virada" não é apenas uma nota matemática, mas um marco de maturidade psicológica e pedagógica. Como a base de 2023 não trouxe esta classificação preenchida (Sim/Não), estabelecer um limite de corte baseado na distribuição estatística histórica do IPV (Indicador de Ponto de Virada) permite automatizar esta classificação. Traduzimos assim uma variável contínua (nota) numa regra de negócio binária e acionável.

5. Critério de Indicação para Bolsa baseado no INDE

Porquê: A atribuição de bolsas é estritamente meritocrática, sendo impulsionada pelo Desempenho Académico (IDA) e Engajamento (IEG), que juntos compõem a maior fatia do INDE. Utilizar os níveis hierárquicos mais altos da ONG (Pedras Topázio e Ametista) como gatilho lógico para a coluna Indicado simula com precisão as regras de seleção do mundo real, identificando de forma automática os talentos de alto rendimento.

6. Desmembramento da "Fase Ideal" em Fase e Série Escolar

Porquê: A coluna original agregava duas informações distintas: o nível interno na metodologia da ONG (Fase) e o ano letivo no sistema de ensino tradicional (Série Escolar). Separar estes dados aumenta a granularidade da análise. Passa a ser possível cruzar a "Série Escolar" com a "Idade" para calcular taxas de defasagem escolar externa de forma totalmente independente da progressão interna do aluno na associação.

7. Mapeamento e Padronização do Cabeçalho de Colunas

Porquê: É uma das melhores práticas em Engenharia de Dados. Padronizar os nomes (letras minúsculas, substituição de espaços por underscores, nomenclatura curta) previne erros de sintaxe durante a programação. Mais importante ainda, garante que o dataset de 2023 tenha a mesma exata estrutura dos anos anteriores (como 2022), permitindo concatenações (pd.concat) perfeitas para análises históricas de longo prazo sem duplicação acidental de colunas.

In [447]:
import pandas as pd

# O parâmetro sheet_name=None diz ao Pandas para carregar TUDO
todas_as_folhas = pd.read_excel('data/base_dados_2024.xlsx', sheet_name=None)

# Para ver os nomes de todas as folhas que foram carregadas:
print(todas_as_folhas.keys())

df = todas_as_folhas['PEDE2024']

dict_keys(['PEDE2022', 'PEDE2023', 'PEDE2024'])


# 1. Inglês

In [448]:
# Criando a flag (0 ou 1) para o modelo futuro
df['Tem_Ingles'] = df['Ing'].apply(lambda x: 0 if pd.isna(x) else 1)

# Preenchendo com 0 apenas para viabilizar as contas de ranking/média
df['Ing'] = df['Ing'].fillna(0)

print(f"Alunos com Inglês: {df['Tem_Ingles'].sum()}")
print(f"Alunos sem Inglês: {len(df) - df['Tem_Ingles'].sum()}")

Alunos com Inglês: 474
Alunos sem Inglês: 682


# 2. Apagar colunas desnecessarias





In [449]:
colunas_para_remover = [
    'Rec Av1', 'Rec Av2', 'Rec Psicologia', 
    'Indicado', 'Atingiu PV', 
    'Destaque IEG', 'Destaque IDA', 'Destaque IPV', 'Avaliador6','Avaliador1' ,'Avaliador2' ,'Avaliador3' ,'Avaliador4' , 'Avaliador5', 'Nº Av'
]

# Executando a remoção
df.drop(columns=colunas_para_remover, inplace=True, errors='ignore')

# 3. Criação dos Rankings 2024 (Cg, Cf, Ct)

In [450]:
# Garantir que o INDE 2024 seja numérico (essencial para o rank funcionar)
df['INDE 2024'] = pd.to_numeric(df['INDE 2024'], errors='coerce')

# Ranking Geral (Cg)
# O rank 'min' faz com que empates recebam o menor número (ex: dois 1º lugares)
df['Cg'] = df['INDE 2024'].rank(ascending=False, method='min')

# Ranking por Fase (Cf) - Atualizado para INDE 2024
df['Cf'] = df.groupby('Fase')['INDE 2024'].rank(ascending=False, method='min')

# Ranking por Turma (Ct) - Atualizado para INDE 2024
df['Ct'] = df.groupby('Turma')['INDE 2024'].rank(ascending=False, method='min')

# Visualizando o resultado para os TOP 5 do Geral em 2024
print("\n Top 5 Alunos Geral (2024):")
cols_view = ['Cg', 'Cf', 'Ct', 'Fase', 'Turma', 'INDE 2024']
print(df[cols_view].sort_values(by='Cg').head())


 Top 5 Alunos Geral (2024):
       Cg   Cf   Ct Fase Turma  INDE 2024
332   1.0  1.0  1.0   1M    1M   9.531325
404   2.0  1.0  1.0   2B    2B   9.498581
550   3.0  1.0  1.0   2R    2R   9.464745
1026  4.0  1.0  1.0   7A    7A   9.437925
575   5.0  1.0  1.0   3A    3A   9.417137


In [451]:
# =====================================================================
# AUDITORIA FINAL DE VALORES NULOS
# =====================================================================
print("\n" + "="*50)
print("--- VERIFICAÇÃO DE DADOS FALTANTES ---")

# Conta os nulos por coluna
nulos_por_coluna = df.isnull().sum()

# Filtra apenas as colunas que têm mais de 0 nulos e ordena da maior para a menor
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna >= 0].sort_values(ascending=False)

# Verifica se o filtro encontrou alguma coisa
if colunas_com_nulos.empty:
    print(" SUCESSO! O dataset de 2023 está 100% limpo, sem nenhum valor nulo.")
else:
    print("ATENÇÃO! As seguintes colunas ainda possuem valores nulos:\n")
    
    # Monta uma tabela formatada para exibir o Nome da Coluna, a Quantidade e a %
    tabela_nulos = pd.DataFrame({
        'Quantidade de Nulos': colunas_com_nulos,
        'Porcentagem (%)': (colunas_com_nulos / len(df)) * 100
    })
    
    # Exibe a tabela com 2 casas decimais
    print(tabela_nulos.round(2))

print("="*50 + "\n")


--- VERIFICAÇÃO DE DADOS FALTANTES ---
ATENÇÃO! As seguintes colunas ainda possuem valores nulos:

                       Quantidade de Nulos  Porcentagem (%)
Pedra 20                               965            83.48
Pedra 21                               892            77.16
INDE 22                                684            59.17
Pedra 22                               684            59.17
Pedra 23                               466            40.31
INDE 23                                466            40.31
Por                                    106             9.17
Mat                                    105             9.08
Ct                                     102             8.82
IAA                                    102             8.82
IPV                                    102             8.82
INDE 2024                              102             8.82
IPS                                    102             8.82
Cf                                     102             8.82


# 6. Desmembramento da Fase Ideal e Série Escolar

In [452]:
# O nome na base 2023 é 'Fase Ideal' (com maiúsculas)
df[['fase_limpa', 'serie_escolar']] = df['Fase Ideal'].str.split(r' \(', expand=True)

# Limpeza e tratamento de strings
df['serie_escolar'] = df['serie_escolar'].str.replace(')', '', regex=False).str.strip()
df['fase_limpa'] = df['fase_limpa'].str.strip()

# Tratamento para o grupo de alfabetização (ALFA)
df.loc[df['fase_limpa'].str.contains('ALFA', na=False), 'fase_limpa'] = 'ALFA'
df.loc[df['fase_limpa'] == 'ALFA', 'serie_escolar'] = 'Alfabetização'

print("--- Distribuição de Fases e Séries 2024 ---")
print(df[['fase_limpa', 'serie_escolar']].value_counts())

--- Distribuição de Fases e Séries 2024 ---
fase_limpa  serie_escolar 
Fase 2      5° e 6° ano       281
Fase 3      7° e 8° ano       233
Fase 1      3° e 4° ano       182
Fase 8      Universitários    102
Fase 5      1° EM              96
Fase 4      9° ano             90
Fase 6      2° EM              70
Fase 7      3° EM              53
ALFA        Alfabetização      49
Name: count, dtype: int64


# Auditoria de Regra de Negocio

In [453]:
from sklearn.linear_model import LinearRegression

# Criar um dataframe temporário só com quem tem todas as notas
indicadores = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV']
df_pesos = df.dropna(subset=['INDE 2024'] + indicadores).copy()

# Remover a Fase 8 (Universitários), pois eles costumam ter uma fórmula à parte
df_pesos = df_pesos[~df_pesos['fase_limpa'].astype(str).str.contains('8')]

# Separar as Variáveis (X = Indicadores, y = Nota Final)
X = df_pesos[indicadores]
y = df_pesos['INDE 2024']

# Aplicar o Algoritmo (fit_intercept=False força a fórmula a partir de zero, sem viés)
modelo = LinearRegression(fit_intercept=False)
modelo.fit(X, y)

# Visualizar a Mágica: Os coeficientes do modelo são os pesos exatos!
pesos_descobertos = pd.DataFrame({
    'Indicador': X.columns,
    'Peso_Matematico': modelo.coef_,
    'Peso_Arredondado': modelo.coef_.round(1) 
})

print(pesos_descobertos)
print(f"\nSoma total dos pesos arredondados: {pesos_descobertos['Peso_Arredondado'].sum():.2f} (Deveria ser 1.00)")

  Indicador  Peso_Matematico  Peso_Arredondado
0       IAN              0.1               0.1
1       IDA              0.2               0.2
2       IEG              0.2               0.2
3       IAA              0.1               0.1
4       IPS              0.1               0.1
5       IPP              0.1               0.1
6       IPV              0.2               0.2

Soma total dos pesos arredondados: 1.00 (Deveria ser 1.00)


In [454]:
df

,RA,Fase,INDE 2024,Pedra 2024,Turma,Nome Anonimizado,Data de Nasc,Idade,Gênero,Ano ingresso,...,IPV,IAN,Fase Ideal,Defasagem,Escola,Ativo/ Inativo,Ativo/ Inativo.1,Tem_Ingles,fase_limpa,serie_escolar
0,RA-1275,ALFA,7.611367,Ametista,ALFA A - G0/G1,Aluno-1275,2016-07-28 00:00:00,8,Masculino,2024,...,5.446667,10.0,ALFA (1° e 2° ano),0,EE Chácara Florida II,Cursando,Cursando,0,ALFA,Alfabetização
1,RA-1276,ALFA,8.002867,Topázio,ALFA A - G0/G1,Aluno-1276,2016-10-16 00:00:00,8,Feminino,2024,...,7.050000,10.0,ALFA (1° e 2° ano),0,EE Chácara Florida II,Cursando,Cursando,0,ALFA,Alfabetização
2,RA-1277,ALFA,7.952200,Ametista,ALFA A - G0/G1,Aluno-1277,2016-08-16 00:00:00,8,Masculino,2024,...,7.046667,10.0,ALFA (1° e 2° ano),0,EE Dom Pedro Villas Boas de Souza,Cursando,Cursando,0,ALFA,Alfabetização
3,RA-868,ALFA,7.156367,Ametista,ALFA A - G0/G1,Aluno-868,2015-11-08 00:00:00,8,Masculino,2023,...,7.213333,5.0,Fase 1 (3° e 4° ano),-1,EE Chácara Florida II,Cursando,Cursando,0,Fase 1,3° e 4° ano
4,RA-1278,ALFA,5.444200,Quartzo,ALFA A - G0/G1,Aluno-1278,2015-03-22 00:00:00,9,Masculino,2024,...,4.173333,5.0,Fase 1 (3° e 4° ano),-1,EM Etelvina Delfim Simões,Cursando,Cursando,0,Fase 1,3° e 4° ano
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1151,RA-1658,9,NaN,INCLUIR,9,Aluno-1658,2002-12-14 02:00:00,21,Masculino,2021,...,NaN,10.0,Fase 8 (Universitários),1,Faculdade (FIAP),Cursando,Cursando,0,Fase 8,Universitários
1152,RA-1659,9,NaN,INCLUIR,9,Aluno-1659,2003-02-04 02:00:00,21,Masculino,2021,...,NaN,10.0,Fase 8 (Universitários),1,Bolsista Universitário *Formado (a),Cursando,Cursando,0,Fase 8,Universitários
1153,RA-1252,9,NaN,INCLUIR,9,Aluno-1252,2002-06-03 03:00:00,22,Feminino,2021,...,NaN,10.0,Fase 8 (Universitários),1,Faculdade (FIAP),Cursando,Cursando,0,Fase 8,Universitários
1154,RA-1660,9,NaN,INCLUIR,9,Aluno-1660,2000-06-28 03:00:00,24,Feminino,2021,...,NaN,10.0,Fase 8 (Universitários),1,Bolsista Universitário *Formado (a),Cursando,Cursando,0,Fase 8,Universitários


# Mapeamento e Padronização de Colunas

In [455]:
# Mapeamento 2024 com sufixo identificador
map_24 = {
    'RA': 'ra', 'Nome Anonimizado': 'nome', 'Gênero': 'genero', 'Idade': 'idade',
    'Ano ingresso': 'ano_ingresso', 'fase': 'fase', 'turma': 'turma',
    'serie_escolar_24': 'serie_escolar', 'ponto_virada_24': 'ponto_virada', 
    'INDE 2024': 'inde',     
    'Pedra 2024': 'pedra',   
    'INDE 23': 'inde_2023',
    'Pedra 23': 'pedra_2023',
    'INDE 22': 'inde_2022',
    'Pedra 22': 'pedra_2022',
    'Pedra 21': 'pedra_2021',
    'Pedra 20': 'pedra_2020',
    'nota_mat': 'mat', 'nota_port': 'por', 'nota_ing': 'ing', 
    'Defasagem': 'defas',
    'fase_limpa' : 'fase_ideal', 
    'Instituição de ensino': 'instituicao_de_ensino',
}

# 1. Aplicar a renomeação
df = df.rename(columns=map_24)

# 2. Normalização residual (snake_case para colunas de histórico ou administrativas)
df.columns = [
    col.lower().replace(' ', '_').replace('.', '_').replace('/', '_') 
    for col in df.columns
]

# 3. Verificação
print("--- Colunas 2024 com Sufixo Ativado ---")
print([c for c in df.columns if c.endswith('_24')])

--- Colunas 2024 com Sufixo Ativado ---
[]


In [456]:
# =====================================================================
# AUDITORIA FINAL DE VALORES NULOS
# =====================================================================
print("\n" + "="*50)
print("--- VERIFICAÇÃO DE DADOS FALTANTES ---")

# Conta os nulos por coluna
nulos_por_coluna = df.isnull().sum()

# Filtra apenas as colunas que têm mais de 0 nulos e ordena da maior para a menor
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna >= 0].sort_values(ascending=False)

# Verifica se o filtro encontrou alguma coisa
if colunas_com_nulos.empty:
    print(" SUCESSO! O dataset de 2024 está 100% limpo, sem nenhum valor nulo.")
else:
    print("ATENÇÃO! As seguintes colunas ainda possuem valores nulos:\n")
    
    # Monta uma tabela formatada para exibir o Nome da Coluna, a Quantidade e a %
    tabela_nulos = pd.DataFrame({
        'Quantidade de Nulos': colunas_com_nulos,
        'Porcentagem (%)': (colunas_com_nulos / len(df)) * 100
    })
    
    # Exibe a tabela com 2 casas decimais
    print(tabela_nulos.round(2))

print("="*50 + "\n")


--- VERIFICAÇÃO DE DADOS FALTANTES ---
ATENÇÃO! As seguintes colunas ainda possuem valores nulos:

                       Quantidade de Nulos  Porcentagem (%)
pedra_2020                             965            83.48
pedra_2021                             892            77.16
inde_2022                              684            59.17
pedra_2022                             684            59.17
inde_2023                              466            40.31
pedra_2023                             466            40.31
por                                    106             9.17
mat                                    105             9.08
iaa                                    102             8.82
ct                                     102             8.82
ips                                    102             8.82
inde                                   102             8.82
cg                                     102             8.82
ipv                                    102             8.82


In [457]:
#Criar a coluna status_aluno: 1 para 'Cursando' (ativo) e 0 para o resto
df['status_aluno'] = df['ativo__inativo'].apply(lambda x: 1 if str(x).lower() == 'cursando' else 0)

# Apagar as duas colunas redundantes
df = df.drop(columns=['ativo__inativo', 'ativo__inativo_1'])

In [458]:
# Normalização residual (lower case e underscore)
df.columns = [col.lower().replace(' ', '_') for col in df.columns]

# Adicionar flag de ano para o EDA histórico
df['ano_referencia'] = 2024

print("\n--- Colunas Padronizadas para 2024 ---")
print(df.columns.tolist())

# Salvar versão final
df.to_csv('pede_2024_final.csv', index=False, encoding='utf-8-sig')
print("\n Base 2024 Padronizada e Salva!")


--- Colunas Padronizadas para 2024 ---
['ra', 'fase', 'inde', 'pedra', 'turma', 'nome', 'data_de_nasc', 'idade', 'genero', 'ano_ingresso', 'instituicao_de_ensino', 'pedra_2020', 'pedra_2021', 'pedra_2022', 'pedra_2023', 'inde_2022', 'inde_2023', 'cg', 'cf', 'ct', 'iaa', 'ieg', 'ips', 'ipp', 'ida', 'mat', 'por', 'ing', 'ipv', 'ian', 'fase_ideal', 'defas', 'escola', 'tem_ingles', 'fase_ideal', 'serie_escolar', 'status_aluno', 'ano_referencia']

 Base 2024 Padronizada e Salva!
